# Actualización de subtipos

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Configuración general de visualización de pandas
pd.set_option(
    "display.max_columns",
    None,
)

pd.set_option(
    "display.max_rows",
    200,
)

pd.set_option(
    "display.width",
    200,
)

pd.set_option(
    "display.float_format",
    lambda valor: f"{valor:,.6f}",
)


# Configuración general de gráficos
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

In [3]:
# ============================================================
# Rutas de entrada y salida
# ============================================================

# ------------------------------------------------------------
# Archivos definitivos construidos antes de aplicar la nueva
# regla de selección de observaciones discordantes
# ------------------------------------------------------------

ruta_catalogo_entrada = Path(
    "../data/interim/catalogo_maestro_final.csv"
)

ruta_curvas_entrada = Path(
    "../data/interim/curvas_plegadas_final.parquet"
)

directorio_discretizaciones_entrada = Path(
    "../data/processed/discretizaciones"
)

rutas_datasets_entrada = {
    ("medianas_fase", 50): (
        directorio_discretizaciones_entrada
        / "medianas_fase_50.parquet"
    ),
    ("medianas_fase", 100): (
        directorio_discretizaciones_entrada
        / "medianas_fase_100.parquet"
    ),
    ("interpolacion", 50): (
        directorio_discretizaciones_entrada
        / "interpolacion_50.parquet"
    ),
    ("interpolacion", 100): (
        directorio_discretizaciones_entrada
        / "interpolacion_100.parquet"
    ),
}

rutas_auditorias_entrada = {
    "medianas_fase": (
        directorio_discretizaciones_entrada
        / "auditoria_medianas_fase.parquet"
    ),
    "interpolacion": (
        directorio_discretizaciones_entrada
        / "auditoria_interpolacion.parquet"
    ),
}


# ------------------------------------------------------------
# Directorio provisional para los productos filtrados
#
# Los archivos actuales no serán sobrescritos hasta comprobar
# completamente la nueva selección.
# ------------------------------------------------------------

directorio_salida = Path(
    "../data/processed/seleccion_ogleiv"
)

ruta_catalogo_salida = (
    directorio_salida
    / "catalogo_maestro_seleccion_ogleiv.csv"
)

ruta_curvas_salida = (
    directorio_salida
    / "curvas_plegadas_seleccion_ogleiv.parquet"
)

rutas_datasets_salida = {
    ("medianas_fase", 50): (
        directorio_salida
        / "medianas_fase_50.parquet"
    ),
    ("medianas_fase", 100): (
        directorio_salida
        / "medianas_fase_100.parquet"
    ),
    ("interpolacion", 50): (
        directorio_salida
        / "interpolacion_50.parquet"
    ),
    ("interpolacion", 100): (
        directorio_salida
        / "interpolacion_100.parquet"
    ),
}

rutas_auditorias_salida = {
    "medianas_fase": (
        directorio_salida
        / "auditoria_medianas_fase.parquet"
    ),
    "interpolacion": (
        directorio_salida
        / "auditoria_interpolacion.parquet"
    ),
}

ruta_auditoria_seleccion = (
    directorio_salida
    / "auditoria_seleccion_ogleiv.csv"
)


# ------------------------------------------------------------
# Verificación de las rutas de entrada
# ------------------------------------------------------------

rutas_entrada = {
    "Catálogo maestro": ruta_catalogo_entrada,
    "Curvas plegadas": ruta_curvas_entrada,
    "Medianas por fase, 50": (
        rutas_datasets_entrada[
            ("medianas_fase", 50)
        ]
    ),
    "Medianas por fase, 100": (
        rutas_datasets_entrada[
            ("medianas_fase", 100)
        ]
    ),
    "Interpolación, 50": (
        rutas_datasets_entrada[
            ("interpolacion", 50)
        ]
    ),
    "Interpolación, 100": (
        rutas_datasets_entrada[
            ("interpolacion", 100)
        ]
    ),
    "Auditoría de medianas": (
        rutas_auditorias_entrada[
            "medianas_fase"
        ]
    ),
    "Auditoría de interpolación": (
        rutas_auditorias_entrada[
            "interpolacion"
        ]
    ),
}

verificacion_rutas = pd.DataFrame([
    {
        "archivo": nombre,
        "ruta": str(ruta.resolve()),
        "encontrado": ruta.exists(),
    }
    for nombre, ruta in rutas_entrada.items()
])

display(
    verificacion_rutas
)

rutas_faltantes = [
    nombre
    for nombre, ruta in rutas_entrada.items()
    if not ruta.exists()
]

if rutas_faltantes:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos de entrada: "
        f"{rutas_faltantes}"
    )

print(
    "Todos los archivos de entrada fueron encontrados."
)

print(
    "Directorio provisional de salida:",
    directorio_salida.resolve(),
)

,archivo,ruta,encontrado
0,Catálogo maestro,C:\Users\adanm\Code\tesis\data\interim\catalog...,True
1,Curvas plegadas,C:\Users\adanm\Code\tesis\data\interim\curvas_...,True
2,"Medianas por fase, 50",C:\Users\adanm\Code\tesis\data\processed\discr...,True
3,"Medianas por fase, 100",C:\Users\adanm\Code\tesis\data\processed\discr...,True
4,"Interpolación, 50",C:\Users\adanm\Code\tesis\data\processed\discr...,True
5,"Interpolación, 100",C:\Users\adanm\Code\tesis\data\processed\discr...,True
6,Auditoría de medianas,C:\Users\adanm\Code\tesis\data\processed\discr...,True
7,Auditoría de interpolación,C:\Users\adanm\Code\tesis\data\processed\discr...,True


Todos los archivos de entrada fueron encontrados.
Directorio provisional de salida: C:\Users\adanm\Code\tesis\data\processed\seleccion_ogleiv


In [4]:
# ============================================================
# Auditoría de observaciones con subtipos discordantes
# ============================================================

# Cargamos el catálogo completo sin inferencia fragmentada
# de tipos, evitando advertencias por columnas mixtas.
catalogo = pd.read_csv(
    ruta_catalogo_entrada,
    low_memory=False,
)

columnas_requeridas = {
    "observacion_id",
    "star_id_base",
    "proyecto_origen",
    "subtipo_rrlyrae",
}

columnas_faltantes = (
    columnas_requeridas
    - set(catalogo.columns)
)

if columnas_faltantes:
    raise ValueError(
        "El catálogo no contiene las columnas requeridas: "
        f"{sorted(columnas_faltantes)}"
    )


# ------------------------------------------------------------
# Normalización de las columnas utilizadas en la auditoría
# ------------------------------------------------------------

for columna in [
    "observacion_id",
    "star_id_base",
    "proyecto_origen",
    "subtipo_rrlyrae",
]:
    catalogo[columna] = (
        catalogo[columna]
        .astype("string")
        .str.strip()
    )


# ------------------------------------------------------------
# Validaciones estructurales iniciales
# ------------------------------------------------------------

ids_nulos = int(
    catalogo["observacion_id"].isna().sum()
)

ids_duplicados = int(
    catalogo["observacion_id"].duplicated().sum()
)

estrellas_nulas = int(
    catalogo["star_id_base"].isna().sum()
)

if ids_nulos > 0:
    raise ValueError(
        f"Se encontraron {ids_nulos} identificadores nulos."
    )

if ids_duplicados > 0:
    raise ValueError(
        f"Se encontraron {ids_duplicados} identificadores duplicados."
    )

if estrellas_nulas > 0:
    raise ValueError(
        f"Se encontraron {estrellas_nulas} valores nulos "
        "en 'star_id_base'."
    )


# ------------------------------------------------------------
# Separación de las observaciones por proyecto
# ------------------------------------------------------------

catalogo_ogleiii = (
    catalogo.loc[
        catalogo["proyecto_origen"] == "OGLE-III",
        [
            "star_id_base",
            "observacion_id",
            "subtipo_rrlyrae",
        ],
    ]
    .rename(columns={
        "observacion_id": "observacion_id_ogleiii",
        "subtipo_rrlyrae": "subtipo_ogleiii_auditado",
    })
    .copy()
)

catalogo_ogleiv = (
    catalogo.loc[
        catalogo["proyecto_origen"] == "OGLE-IV",
        [
            "star_id_base",
            "observacion_id",
            "subtipo_rrlyrae",
        ],
    ]
    .rename(columns={
        "observacion_id": "observacion_id_ogleiv",
        "subtipo_rrlyrae": "subtipo_ogleiv_auditado",
    })
    .copy()
)


# Cada estrella debe tener como máximo una observación
# dentro de cada proyecto.
duplicados_ogleiii = int(
    catalogo_ogleiii["star_id_base"]
    .duplicated()
    .sum()
)

duplicados_ogleiv = int(
    catalogo_ogleiv["star_id_base"]
    .duplicated()
    .sum()
)

if duplicados_ogleiii > 0:
    raise ValueError(
        "Se encontraron estrellas con más de una observación "
        f"en OGLE-III: {duplicados_ogleiii}"
    )

if duplicados_ogleiv > 0:
    raise ValueError(
        "Se encontraron estrellas con más de una observación "
        f"en OGLE-IV: {duplicados_ogleiv}"
    )


# ------------------------------------------------------------
# Construcción de los pares OGLE-III / OGLE-IV
# ------------------------------------------------------------

pares_ogle = catalogo_ogleiii.merge(
    catalogo_ogleiv,
    on="star_id_base",
    how="inner",
    validate="one_to_one",
)

pares_ogle["subtipos_coincidentes"] = (
    pares_ogle["subtipo_ogleiii_auditado"]
    == pares_ogle["subtipo_ogleiv_auditado"]
)

pares_ogle["transicion_auditada"] = (
    pares_ogle["subtipo_ogleiii_auditado"]
    + " → "
    + pares_ogle["subtipo_ogleiv_auditado"]
)

pares_coincidentes = (
    pares_ogle.loc[
        pares_ogle["subtipos_coincidentes"]
    ]
    .copy()
)

pares_discordantes = (
    pares_ogle.loc[
        ~pares_ogle["subtipos_coincidentes"]
    ]
    .copy()
)


# ------------------------------------------------------------
# Identificadores que se retirarían
#
# Solamente se retira la observación OGLE-III de cada par
# discordante. La observación OGLE-IV se conserva.
# ------------------------------------------------------------

ids_retirar = set(
    pares_discordantes[
        "observacion_id_ogleiii"
    ].tolist()
)

ids_ogleiv_conservar = set(
    pares_discordantes[
        "observacion_id_ogleiv"
    ].tolist()
)

interseccion_seleccion = (
    ids_retirar
    & ids_ogleiv_conservar
)

if interseccion_seleccion:
    raise ValueError(
        "Existen identificadores marcados simultáneamente "
        "para retirar y conservar."
    )


# ------------------------------------------------------------
# Simulación de la selección en memoria
#
# Todavía no se escribe ningún archivo.
# ------------------------------------------------------------

catalogo_seleccion = (
    catalogo.loc[
        ~catalogo["observacion_id"].isin(
            ids_retirar
        )
    ]
    .copy()
)

estrellas_originales = int(
    catalogo["star_id_base"].nunique()
)

estrellas_finales_esperadas = int(
    catalogo_seleccion["star_id_base"].nunique()
)

contrapartes_ogleiv_faltantes = int(
    len(
        ids_ogleiv_conservar
        - set(
            catalogo_seleccion[
                "observacion_id"
            ]
        )
    )
)

discordantes_ogleiii_restantes = int(
    catalogo_seleccion[
        "observacion_id"
    ]
    .isin(ids_retirar)
    .sum()
)


# ------------------------------------------------------------
# Cálculo provisional de las mediciones que se retirarían
# ------------------------------------------------------------

columna_mediciones = None

for candidata in [
    "n_obs_folding",
    "n_obs_validas",
    "n_obs",
]:
    if candidata in catalogo.columns:
        columna_mediciones = candidata
        break

if columna_mediciones is not None:
    mediciones_retiradas_estimadas = int(
        pd.to_numeric(
            catalogo.loc[
                catalogo["observacion_id"].isin(
                    ids_retirar
                ),
                columna_mediciones,
            ],
            errors="coerce",
        )
        .fillna(0)
        .sum()
    )
else:
    mediciones_retiradas_estimadas = np.nan


# ------------------------------------------------------------
# Revisión de las dos curvas con anomalías fotométricas
# ------------------------------------------------------------

ids_anomalias_fotometricas = {
    "OGLEIII_OGLE-LMC-RRLYR-14167",
    "OGLEIII_OGLE-LMC-RRLYR-17594",
}

revision_anomalias = (
    catalogo.loc[
        catalogo["observacion_id"].isin(
            ids_anomalias_fotometricas
        ),
        [
            "observacion_id",
            "star_id_base",
            "proyecto_origen",
            "subtipo_rrlyrae",
        ],
    ]
    .copy()
)

revision_anomalias["se_retiraria"] = (
    revision_anomalias["observacion_id"]
    .isin(ids_retirar)
)


# ------------------------------------------------------------
# Resumen principal de la auditoría
# ------------------------------------------------------------

resumen_auditoria = pd.DataFrame([
    {
        "metrica": "Filas actuales del catálogo",
        "valor": len(catalogo),
    },
    {
        "metrica": "Estrellas físicas actuales",
        "valor": estrellas_originales,
    },
    {
        "metrica": "Pares OGLE-III / OGLE-IV",
        "valor": len(pares_ogle),
    },
    {
        "metrica": "Pares con subtipo coincidente",
        "valor": len(pares_coincidentes),
    },
    {
        "metrica": "Pares con subtipo discordante",
        "valor": len(pares_discordantes),
    },
    {
        "metrica": "Observaciones OGLE-III para retirar",
        "valor": len(ids_retirar),
    },
    {
        "metrica": "Observaciones OGLE-IV conservadas",
        "valor": len(ids_ogleiv_conservar),
    },
    {
        "metrica": "Filas esperadas después de seleccionar",
        "valor": len(catalogo_seleccion),
    },
    {
        "metrica": "Estrellas físicas esperadas",
        "valor": estrellas_finales_esperadas,
    },
    {
        "metrica": "Contrapartes OGLE-IV faltantes",
        "valor": contrapartes_ogleiv_faltantes,
    },
    {
        "metrica": "Discordantes OGLE-III restantes",
        "valor": discordantes_ogleiii_restantes,
    },
    {
        "metrica": (
            "Mediciones estimadas para retirar"
        ),
        "valor": mediciones_retiradas_estimadas,
    },
])


# ------------------------------------------------------------
# Resumen de transiciones discordantes
# ------------------------------------------------------------

resumen_transiciones = (
    pares_discordantes[
        "transicion_auditada"
    ]
    .value_counts()
    .rename_axis("transicion")
    .reset_index(name="cantidad")
)


display(
    resumen_auditoria
)

display(
    resumen_transiciones
)

display(
    revision_anomalias
)

print(
    "Columna utilizada para estimar mediciones:",
    columna_mediciones,
)

print(
    "Esta celda no creó ni modificó archivos."
)

,metrica,valor
0,Filas actuales del catálogo,104008
1,Estrellas físicas actuales,70269
2,Pares OGLE-III / OGLE-IV,33739
3,Pares con subtipo coincidente,32509
4,Pares con subtipo discordante,1230
5,Observaciones OGLE-III para retirar,1230
6,Observaciones OGLE-IV conservadas,1230
7,Filas esperadas después de seleccionar,102778
8,Estrellas físicas esperadas,70269
9,Contrapartes OGLE-IV faltantes,0


,transicion,cantidad
0,RRe → RRc,1175
1,RRc → RRd,16
2,RRab → aRRd,11
3,RRd → RRab,8
4,RRab → RRd,5
5,RRc → RRab,4
6,RRd → RRc,4
7,RRc → aRRd,4
8,RRab → RRc,3


,observacion_id,star_id_base,proyecto_origen,subtipo_rrlyrae,se_retiraria
13650,OGLEIII_OGLE-LMC-RRLYR-14167,OGLE-LMC-RRLYR-14167,OGLE-III,RRe,True
16974,OGLEIII_OGLE-LMC-RRLYR-17594,OGLE-LMC-RRLYR-17594,OGLE-III,RRab,False


Columna utilizada para estimar mediciones: n_obs_folding
Esta celda no creó ni modificó archivos.


In [5]:
# ============================================================
# Auditoría de los identificadores que se retirarán en todos
# los productos construidos
# ============================================================

# ------------------------------------------------------------
# Revisión directa del Parquet de curvas plegadas
# ------------------------------------------------------------

archivo_curvas = pq.ParquetFile(
    ruta_curvas_entrada
)

ids_curvas_plegadas = set()

filas_curvas_actuales = 0
mediciones_curvas_actuales = 0

filas_curvas_retirar = 0
mediciones_curvas_retirar = 0

for numero_grupo in range(
    archivo_curvas.num_row_groups
):
    tabla_grupo = archivo_curvas.read_row_group(
        numero_grupo,
        columns=[
            "observacion_id",
            "fase",
        ],
    )

    grupo = tabla_grupo.to_pydict()

    for observacion_id, fases in zip(
        grupo["observacion_id"],
        grupo["fase"],
    ):
        numero_mediciones = len(fases)

        ids_curvas_plegadas.add(
            observacion_id
        )

        filas_curvas_actuales += 1
        mediciones_curvas_actuales += (
            numero_mediciones
        )

        if observacion_id in ids_retirar:
            filas_curvas_retirar += 1
            mediciones_curvas_retirar += (
                numero_mediciones
            )


filas_curvas_esperadas = (
    filas_curvas_actuales
    - filas_curvas_retirar
)

mediciones_curvas_esperadas = (
    mediciones_curvas_actuales
    - mediciones_curvas_retirar
)

ids_retirar_faltantes_curvas = (
    ids_retirar
    - ids_curvas_plegadas
)


# ------------------------------------------------------------
# Revisión de los cuatro datasets discretizados
# ------------------------------------------------------------

auditoria_datasets = []

ids_catalogo_actual = set(
    catalogo["observacion_id"]
)

for (
    metodo,
    resolucion,
), ruta_dataset in rutas_datasets_entrada.items():

    tabla_ids = pq.read_table(
        ruta_dataset,
        columns=["observacion_id"],
    )

    ids_dataset_lista = (
        tabla_ids[
            "observacion_id"
        ]
        .to_pylist()
    )

    ids_dataset = set(
        ids_dataset_lista
    )

    ids_retirar_en_dataset = (
        ids_dataset
        & ids_retirar
    )

    auditoria_datasets.append({
        "producto": (
            f"{metodo}_{resolucion}"
        ),
        "filas_actuales": len(
            ids_dataset_lista
        ),
        "ids_unicos": len(
            ids_dataset
        ),
        "ids_retirar_encontrados": len(
            ids_retirar_en_dataset
        ),
        "ids_retirar_faltantes": len(
            ids_retirar
            - ids_dataset
        ),
        "ids_faltantes_catalogo": len(
            ids_catalogo_actual
            - ids_dataset
        ),
        "ids_adicionales": len(
            ids_dataset
            - ids_catalogo_actual
        ),
        "filas_esperadas": (
            len(ids_dataset_lista)
            - len(ids_retirar_en_dataset)
        ),
    })

auditoria_datasets = pd.DataFrame(
    auditoria_datasets
)


# ------------------------------------------------------------
# Revisión de las auditorías de discretización
# ------------------------------------------------------------

auditoria_archivos_auditoria = []

for nombre, ruta_auditoria in (
    rutas_auditorias_entrada.items()
):

    tabla_ids = pq.read_table(
        ruta_auditoria,
        columns=["observacion_id"],
    )

    ids_auditoria_lista = (
        tabla_ids[
            "observacion_id"
        ]
        .to_pylist()
    )

    mascara_retirar = [
        observacion_id in ids_retirar
        for observacion_id
        in ids_auditoria_lista
    ]

    filas_retirar_auditoria = int(
        np.sum(mascara_retirar)
    )

    ids_retirar_auditoria = {
        observacion_id
        for observacion_id
        in ids_auditoria_lista
        if observacion_id in ids_retirar
    }

    auditoria_archivos_auditoria.append({
        "auditoria": nombre,
        "filas_actuales": len(
            ids_auditoria_lista
        ),
        "observaciones_unicas": len(
            set(ids_auditoria_lista)
        ),
        "filas_para_retirar": (
            filas_retirar_auditoria
        ),
        "ids_retirar_encontrados": len(
            ids_retirar_auditoria
        ),
        "ids_retirar_faltantes": len(
            ids_retirar
            - ids_retirar_auditoria
        ),
        "filas_esperadas": (
            len(ids_auditoria_lista)
            - filas_retirar_auditoria
        ),
    })

auditoria_archivos_auditoria = pd.DataFrame(
    auditoria_archivos_auditoria
)


# ------------------------------------------------------------
# Comparación entre el catálogo y el Parquet
# ------------------------------------------------------------

mediciones_catalogo_actual = int(
    pd.to_numeric(
        catalogo[columna_mediciones],
        errors="coerce",
    )
    .fillna(0)
    .sum()
)

mediciones_catalogo_retirar = int(
    pd.to_numeric(
        catalogo.loc[
            catalogo["observacion_id"].isin(
                ids_retirar
            ),
            columna_mediciones,
        ],
        errors="coerce",
    )
    .fillna(0)
    .sum()
)

mediciones_catalogo_esperadas = (
    mediciones_catalogo_actual
    - mediciones_catalogo_retirar
)

resumen_productos = pd.DataFrame([
    {
        "metrica": "Curvas actuales en el Parquet",
        "valor": filas_curvas_actuales,
    },
    {
        "metrica": "Curvas para retirar del Parquet",
        "valor": filas_curvas_retirar,
    },
    {
        "metrica": "Curvas esperadas",
        "valor": filas_curvas_esperadas,
    },
    {
        "metrica": "Mediciones actuales en el Parquet",
        "valor": mediciones_curvas_actuales,
    },
    {
        "metrica": "Mediciones para retirar",
        "valor": mediciones_curvas_retirar,
    },
    {
        "metrica": "Mediciones finales esperadas",
        "valor": mediciones_curvas_esperadas,
    },
    {
        "metrica": "Mediciones actuales según catálogo",
        "valor": mediciones_catalogo_actual,
    },
    {
        "metrica": "Mediciones para retirar según catálogo",
        "valor": mediciones_catalogo_retirar,
    },
    {
        "metrica": "Mediciones finales según catálogo",
        "valor": mediciones_catalogo_esperadas,
    },
    {
        "metrica": "IDs para retirar faltantes en el Parquet",
        "valor": len(
            ids_retirar_faltantes_curvas
        ),
    },
    {
        "metrica": "Coincidencia actual catálogo-Parquet",
        "valor": (
            mediciones_catalogo_actual
            == mediciones_curvas_actuales
        ),
    },
    {
        "metrica": "Coincidencia retirada catálogo-Parquet",
        "valor": (
            mediciones_catalogo_retirar
            == mediciones_curvas_retirar
        ),
    },
    {
        "metrica": "Coincidencia final catálogo-Parquet",
        "valor": (
            mediciones_catalogo_esperadas
            == mediciones_curvas_esperadas
        ),
    },
])


display(
    resumen_productos
)

display(
    auditoria_datasets
)

display(
    auditoria_archivos_auditoria
)

print(
    "Esta celda no creó ni modificó archivos."
)

,metrica,valor
0,Curvas actuales en el Parquet,104008
1,Curvas para retirar del Parquet,1230
2,Curvas esperadas,102778
3,Mediciones actuales en el Parquet,132232695
4,Mediciones para retirar,692018
5,Mediciones finales esperadas,131540677
6,Mediciones actuales según catálogo,132232695
7,Mediciones para retirar según catálogo,692018
8,Mediciones finales según catálogo,131540677
9,IDs para retirar faltantes en el Parquet,0


,producto,filas_actuales,ids_unicos,ids_retirar_encontrados,ids_retirar_faltantes,ids_faltantes_catalogo,ids_adicionales,filas_esperadas
0,medianas_fase_50,104008,104008,1230,0,0,0,102778
1,medianas_fase_100,104008,104008,1230,0,0,0,102778
2,interpolacion_50,104008,104008,1230,0,0,0,102778
3,interpolacion_100,104008,104008,1230,0,0,0,102778


,auditoria,filas_actuales,observaciones_unicas,filas_para_retirar,ids_retirar_encontrados,ids_retirar_faltantes,filas_esperadas
0,medianas_fase,208016,104008,2460,1230,0,205556
1,interpolacion,104008,104008,1230,1230,0,102778


Esta celda no creó ni modificó archivos.


In [6]:
# ============================================================
# Generación de productos con prioridad de OGLE-IV
# ============================================================

# ------------------------------------------------------------
# Comprobación de seguridad
# ------------------------------------------------------------

rutas_salida = [
    ruta_catalogo_salida,
    ruta_curvas_salida,
    ruta_auditoria_seleccion,
    *rutas_datasets_salida.values(),
    *rutas_auditorias_salida.values(),
]

archivos_salida_existentes = [
    ruta
    for ruta in rutas_salida
    if ruta.exists()
]

if archivos_salida_existentes:
    detalle = "\n".join(
        str(ruta.resolve())
        for ruta in archivos_salida_existentes
    )

    raise FileExistsError(
        "La generación se detuvo porque ya existen "
        "archivos de salida:\n"
        f"{detalle}"
    )

directorio_salida.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Preparación del catálogo final
# ------------------------------------------------------------

catalogo_salida = (
    catalogo.loc[
        ~catalogo["observacion_id"].isin(
            ids_retirar
        )
    ]
    .copy()
)

# Recalculamos esta variable a partir de las filas que realmente
# permanecen en el catálogo. Una estrella tiene dos observaciones
# solo si conserva una curva de OGLE-III y otra de OGLE-IV.
numero_observaciones_por_estrella = (
    catalogo_salida
    .groupby("star_id_base")[
        "observacion_id"
    ]
    .transform("size")
)

catalogo_salida["dos_observaciones"] = (
    numero_observaciones_por_estrella == 2
)


# Reasignamos el grupo de análisis utilizando exclusivamente
# el subtipo vigente de cada observación.
subtipos_principales = {
    "RRab",
    "RRc",
    "RRd",
}

subtipos_minoritarios = {
    "RRe",
    "aRRd",
}

subtipos_presentes = set(
    catalogo_salida[
        "subtipo_rrlyrae"
    ]
    .dropna()
    .unique()
)

subtipos_no_reconocidos = (
    subtipos_presentes
    - subtipos_principales
    - subtipos_minoritarios
)

if subtipos_no_reconocidos:
    raise ValueError(
        "Se encontraron subtipos sin grupo definido: "
        f"{sorted(subtipos_no_reconocidos)}"
    )

catalogo_salida["grupo_analisis"] = np.select(
    [
        catalogo_salida[
            "subtipo_rrlyrae"
        ].isin(subtipos_principales),
        catalogo_salida[
            "subtipo_rrlyrae"
        ].isin(subtipos_minoritarios),
    ],
    [
        "principal",
        "minoritaria",
    ],
    default="sin_clasificar",
)


# Eliminamos las columnas que describían la comparación histórica
# entre los subtipos de OGLE-III y OGLE-IV.
columnas_eliminar = [
    "subtipos_discordantes",
    "transicion",
    "subtipo_ogleiii",
    "subtipo_ogleiv",
]

columnas_eliminar_presentes = [
    columna
    for columna in columnas_eliminar
    if columna in catalogo_salida.columns
]

catalogo_salida = catalogo_salida.drop(
    columns=columnas_eliminar_presentes
)


# ------------------------------------------------------------
# Auditoría externa de la selección
#
# Este archivo conserva la trazabilidad del filtro, pero no forma
# parte del catálogo utilizado en los análisis.
# ------------------------------------------------------------

auditoria_seleccion = (
    pares_discordantes[
        [
            "star_id_base",
            "observacion_id_ogleiii",
            "subtipo_ogleiii_auditado",
            "observacion_id_ogleiv",
            "subtipo_ogleiv_auditado",
            "transicion_auditada",
        ]
    ]
    .copy()
)

auditoria_seleccion["decision"] = (
    "retirar OGLE-III y conservar OGLE-IV"
)


# ------------------------------------------------------------
# Escritura del catálogo y de la auditoría de selección
# ------------------------------------------------------------

catalogo_salida.to_csv(
    ruta_catalogo_salida,
    index=False,
)

auditoria_seleccion.to_csv(
    ruta_auditoria_seleccion,
    index=False,
)


# ------------------------------------------------------------
# Filtrado del Parquet de curvas plegadas por grupos internos
# ------------------------------------------------------------

archivo_curvas_entrada = pq.ParquetFile(
    ruta_curvas_entrada
)

escritor_curvas = pq.ParquetWriter(
    ruta_curvas_salida,
    archivo_curvas_entrada.schema_arrow,
    compression="zstd",
    use_dictionary=True,
)

filas_curvas_escritas = 0
mediciones_curvas_escritas = 0

try:
    for numero_grupo in range(
        archivo_curvas_entrada.num_row_groups
    ):
        tabla_grupo = (
            archivo_curvas_entrada
            .read_row_group(numero_grupo)
        )

        ids_grupo = (
            tabla_grupo[
                "observacion_id"
            ]
            .to_pylist()
        )

        mascara_conservar = pa.array(
            [
                observacion_id
                not in ids_retirar
                for observacion_id in ids_grupo
            ],
            type=pa.bool_(),
        )

        tabla_filtrada = tabla_grupo.filter(
            mascara_conservar
        )

        if tabla_filtrada.num_rows == 0:
            continue

        escritor_curvas.write_table(
            tabla_filtrada
        )

        filas_curvas_escritas += (
            tabla_filtrada.num_rows
        )

        mediciones_curvas_escritas += sum(
            len(fases)
            for fases in tabla_filtrada[
                "fase"
            ].to_pylist()
        )

        if (
            numero_grupo + 1
        ) % 20 == 0:
            print(
                f"{numero_grupo + 1} de "
                f"{archivo_curvas_entrada.num_row_groups} "
                "grupos de curvas procesados"
            )

finally:
    escritor_curvas.close()


# ------------------------------------------------------------
# Filtrado de los cuatro datasets discretizados
# ------------------------------------------------------------

resumen_datasets_escritos = []

for clave, ruta_entrada in (
    rutas_datasets_entrada.items()
):
    metodo, resolucion = clave

    tabla_dataset = pq.read_table(
        ruta_entrada
    )

    ids_dataset = (
        tabla_dataset[
            "observacion_id"
        ]
        .to_pylist()
    )

    mascara_conservar = pa.array(
        [
            observacion_id
            not in ids_retirar
            for observacion_id in ids_dataset
        ],
        type=pa.bool_(),
    )

    tabla_dataset_filtrada = (
        tabla_dataset.filter(
            mascara_conservar
        )
    )

    pq.write_table(
        tabla_dataset_filtrada,
        rutas_datasets_salida[clave],
        compression="zstd",
        use_dictionary=True,
    )

    resumen_datasets_escritos.append({
        "metodo": metodo,
        "resolucion": resolucion,
        "filas_escritas": (
            tabla_dataset_filtrada.num_rows
        ),
        "columnas_escritas": (
            tabla_dataset_filtrada.num_columns
        ),
        "ruta": str(
            rutas_datasets_salida[
                clave
            ].resolve()
        ),
    })

resumen_datasets_escritos = pd.DataFrame(
    resumen_datasets_escritos
)


# ------------------------------------------------------------
# Filtrado de las auditorías de discretización
# ------------------------------------------------------------

resumen_auditorias_escritas = []

for nombre, ruta_entrada in (
    rutas_auditorias_entrada.items()
):
    tabla_auditoria = pq.read_table(
        ruta_entrada
    )

    ids_auditoria = (
        tabla_auditoria[
            "observacion_id"
        ]
        .to_pylist()
    )

    mascara_conservar = pa.array(
        [
            observacion_id
            not in ids_retirar
            for observacion_id in ids_auditoria
        ],
        type=pa.bool_(),
    )

    tabla_auditoria_filtrada = (
        tabla_auditoria.filter(
            mascara_conservar
        )
    )

    pq.write_table(
        tabla_auditoria_filtrada,
        rutas_auditorias_salida[nombre],
        compression="zstd",
        use_dictionary=True,
    )

    resumen_auditorias_escritas.append({
        "auditoria": nombre,
        "filas_escritas": (
            tabla_auditoria_filtrada.num_rows
        ),
        "observaciones_unicas": len(
            set(
                tabla_auditoria_filtrada[
                    "observacion_id"
                ].to_pylist()
            )
        ),
        "ruta": str(
            rutas_auditorias_salida[
                nombre
            ].resolve()
        ),
    })

resumen_auditorias_escritas = pd.DataFrame(
    resumen_auditorias_escritas
)


# ------------------------------------------------------------
# Resumen de los productos creados
# ------------------------------------------------------------

resumen_catalogo_salida = pd.DataFrame([
    {
        "metrica": "Filas del catálogo",
        "valor": len(catalogo_salida),
    },
    {
        "metrica": "Estrellas físicas",
        "valor": (
            catalogo_salida[
                "star_id_base"
            ].nunique()
        ),
    },
    {
        "metrica": "Curvas con dos observaciones",
        "valor": int(
            catalogo_salida.loc[
                catalogo_salida[
                    "dos_observaciones"
                ],
                "star_id_base",
            ].nunique()
        ),
    },
    {
        "metrica": "Curvas plegadas escritas",
        "valor": filas_curvas_escritas,
    },
    {
        "metrica": "Mediciones escritas",
        "valor": mediciones_curvas_escritas,
    },
    {
        "metrica": "Registros de auditoría de selección",
        "valor": len(auditoria_seleccion),
    },
])


display(
    resumen_catalogo_salida
)

display(
    resumen_datasets_escritos
)

display(
    resumen_auditorias_escritas
)

print(
    "Catálogo provisional:",
    ruta_catalogo_salida.resolve(),
)

print(
    "Curvas plegadas provisionales:",
    ruta_curvas_salida.resolve(),
)

print(
    "Auditoría de selección:",
    ruta_auditoria_seleccion.resolve(),
)

20 de 105 grupos de curvas procesados
40 de 105 grupos de curvas procesados
60 de 105 grupos de curvas procesados
80 de 105 grupos de curvas procesados
100 de 105 grupos de curvas procesados


,metrica,valor
0,Filas del catálogo,102778
1,Estrellas físicas,70269
2,Curvas con dos observaciones,32509
3,Curvas plegadas escritas,102778
4,Mediciones escritas,131540677
5,Registros de auditoría de selección,1230


,metodo,resolucion,filas_escritas,columnas_escritas,ruta
0,medianas_fase,50,102778,51,C:\Users\adanm\Code\tesis\data\processed\selec...
1,medianas_fase,100,102778,101,C:\Users\adanm\Code\tesis\data\processed\selec...
2,interpolacion,50,102778,51,C:\Users\adanm\Code\tesis\data\processed\selec...
3,interpolacion,100,102778,101,C:\Users\adanm\Code\tesis\data\processed\selec...


,auditoria,filas_escritas,observaciones_unicas,ruta
0,medianas_fase,205556,102778,C:\Users\adanm\Code\tesis\data\processed\selec...
1,interpolacion,102778,102778,C:\Users\adanm\Code\tesis\data\processed\selec...


Catálogo provisional: C:\Users\adanm\Code\tesis\data\processed\seleccion_ogleiv\catalogo_maestro_seleccion_ogleiv.csv
Curvas plegadas provisionales: C:\Users\adanm\Code\tesis\data\processed\seleccion_ogleiv\curvas_plegadas_seleccion_ogleiv.parquet
Auditoría de selección: C:\Users\adanm\Code\tesis\data\processed\seleccion_ogleiv\auditoria_seleccion_ogleiv.csv


In [7]:
# ============================================================
# Validación integral de los productos provisionales
# ============================================================

# ------------------------------------------------------------
# Reapertura del catálogo desde el archivo escrito
# ------------------------------------------------------------

catalogo_validacion = pd.read_csv(
    ruta_catalogo_salida,
    low_memory=False,
)

for columna in [
    "observacion_id",
    "star_id_base",
    "proyecto_origen",
    "subtipo_rrlyrae",
    "grupo_analisis",
]:
    catalogo_validacion[columna] = (
        catalogo_validacion[columna]
        .astype("string")
        .str.strip()
    )

columnas_que_deben_estar_ausentes = {
    "subtipos_discordantes",
    "transicion",
    "subtipo_ogleiii",
    "subtipo_ogleiv",
}

columnas_eliminadas_presentes = (
    columnas_que_deben_estar_ausentes
    & set(catalogo_validacion.columns)
)


# ------------------------------------------------------------
# Validación de dos_observaciones
# ------------------------------------------------------------

conteo_por_estrella = (
    catalogo_validacion
    .groupby("star_id_base")[
        "observacion_id"
    ]
    .transform("size")
)

dos_observaciones_esperado = (
    conteo_por_estrella == 2
)

dos_observaciones_catalogo = (
    catalogo_validacion[
        "dos_observaciones"
    ]
    .astype("boolean")
)

diferencias_dos_observaciones = int(
    (
        dos_observaciones_catalogo
        != dos_observaciones_esperado
    )
    .fillna(True)
    .sum()
)


# ------------------------------------------------------------
# Validación de grupo_analisis
# ------------------------------------------------------------

grupo_esperado = np.select(
    [
        catalogo_validacion[
            "subtipo_rrlyrae"
        ].isin({
            "RRab",
            "RRc",
            "RRd",
        }),
        catalogo_validacion[
            "subtipo_rrlyrae"
        ].isin({
            "RRe",
            "aRRd",
        }),
    ],
    [
        "principal",
        "minoritaria",
    ],
    default="sin_clasificar",
)

diferencias_grupo_analisis = int(
    (
        catalogo_validacion[
            "grupo_analisis"
        ].to_numpy()
        != grupo_esperado
    ).sum()
)


# ------------------------------------------------------------
# Verificación de pares OGLE-III / OGLE-IV restantes
# ------------------------------------------------------------

catalogo_final_ogleiii = (
    catalogo_validacion.loc[
        catalogo_validacion[
            "proyecto_origen"
        ] == "OGLE-III",
        [
            "star_id_base",
            "subtipo_rrlyrae",
        ],
    ]
    .rename(columns={
        "subtipo_rrlyrae": "subtipo_ogleiii",
    })
)

catalogo_final_ogleiv = (
    catalogo_validacion.loc[
        catalogo_validacion[
            "proyecto_origen"
        ] == "OGLE-IV",
        [
            "star_id_base",
            "subtipo_rrlyrae",
        ],
    ]
    .rename(columns={
        "subtipo_rrlyrae": "subtipo_ogleiv",
    })
)

pares_finales = (
    catalogo_final_ogleiii
    .merge(
        catalogo_final_ogleiv,
        on="star_id_base",
        how="inner",
        validate="one_to_one",
    )
)

pares_finales_discordantes = int(
    (
        pares_finales["subtipo_ogleiii"]
        != pares_finales["subtipo_ogleiv"]
    ).sum()
)


# ------------------------------------------------------------
# Reapertura y validación del Parquet de curvas plegadas
# ------------------------------------------------------------

archivo_curvas_validacion = pq.ParquetFile(
    ruta_curvas_salida
)

columnas_curvas = set(
    archivo_curvas_validacion
    .schema_arrow
    .names
)

usar_conteo_almacenado = (
    "n_obs_folding"
    in columnas_curvas
)

columnas_lectura_curvas = [
    "observacion_id",
]

if usar_conteo_almacenado:
    columnas_lectura_curvas.append(
        "n_obs_folding"
    )
else:
    columnas_lectura_curvas.append(
        "fase"
    )

ids_curvas_validacion = []
mediciones_curvas_validacion = 0

for numero_grupo in range(
    archivo_curvas_validacion.num_row_groups
):
    tabla_grupo = (
        archivo_curvas_validacion
        .read_row_group(
            numero_grupo,
            columns=columnas_lectura_curvas,
        )
    )

    grupo = tabla_grupo.to_pydict()

    ids_curvas_validacion.extend(
        grupo["observacion_id"]
    )

    if usar_conteo_almacenado:
        mediciones_curvas_validacion += sum(
            int(valor)
            for valor in grupo[
                "n_obs_folding"
            ]
        )
    else:
        mediciones_curvas_validacion += sum(
            len(fases)
            for fases in grupo["fase"]
        )

ids_catalogo_validacion = (
    catalogo_validacion[
        "observacion_id"
    ]
    .tolist()
)

set_ids_catalogo_validacion = set(
    ids_catalogo_validacion
)

set_ids_curvas_validacion = set(
    ids_curvas_validacion
)


# ------------------------------------------------------------
# Validación de los cuatro datasets
# ------------------------------------------------------------

validacion_datasets = []

for clave, ruta_dataset in (
    rutas_datasets_salida.items()
):
    metodo, resolucion = clave

    dataset = pd.read_parquet(
        ruta_dataset
    )

    columnas_magnitud = sorted([
        columna
        for columna in dataset.columns
        if columna.startswith(
            "magnitud_"
        )
    ])

    ids_dataset = (
        dataset["observacion_id"]
        .astype("string")
        .tolist()
    )

    matriz = (
        dataset[
            columnas_magnitud
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    validacion_datasets.append({
        "metodo": metodo,
        "resolucion": resolucion,
        "filas": len(dataset),
        "caracteristicas": len(
            columnas_magnitud
        ),
        "ids_unicos": (
            dataset[
                "observacion_id"
            ].nunique()
        ),
        "ids_duplicados": int(
            dataset[
                "observacion_id"
            ]
            .duplicated()
            .sum()
        ),
        "ids_faltantes_catalogo": len(
            set_ids_catalogo_validacion
            - set(ids_dataset)
        ),
        "ids_adicionales": len(
            set(ids_dataset)
            - set_ids_catalogo_validacion
        ),
        "mismo_orden_catalogo": (
            ids_dataset
            == ids_catalogo_validacion
        ),
        "valores_nan": int(
            np.isnan(matriz).sum()
        ),
        "valores_infinitos": int(
            np.isinf(matriz).sum()
        ),
        "valor_minimo": float(
            np.min(matriz)
        ),
        "valor_maximo": float(
            np.max(matriz)
        ),
    })

validacion_datasets = pd.DataFrame(
    validacion_datasets
)


# ------------------------------------------------------------
# Validación de las auditorías filtradas
# ------------------------------------------------------------

auditoria_medianas_validacion = (
    pd.read_parquet(
        rutas_auditorias_salida[
            "medianas_fase"
        ]
    )
)

auditoria_interpolacion_validacion = (
    pd.read_parquet(
        rutas_auditorias_salida[
            "interpolacion"
        ]
    )
)

ids_auditoria_medianas = set(
    auditoria_medianas_validacion[
        "observacion_id"
    ]
)

ids_auditoria_interpolacion = set(
    auditoria_interpolacion_validacion[
        "observacion_id"
    ]
)

if "resolucion" in (
    auditoria_medianas_validacion.columns
):
    pares_medianas_unicos = int(
        auditoria_medianas_validacion[
            [
                "observacion_id",
                "resolucion",
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )

    resoluciones_medianas = sorted(
        auditoria_medianas_validacion[
            "resolucion"
        ]
        .unique()
        .tolist()
    )
else:
    pares_medianas_unicos = np.nan
    resoluciones_medianas = []

resumen_auditorias_validacion = pd.DataFrame([
    {
        "auditoria": "medianas_fase",
        "filas": len(
            auditoria_medianas_validacion
        ),
        "observaciones_unicas": len(
            ids_auditoria_medianas
        ),
        "ids_faltantes_catalogo": len(
            set_ids_catalogo_validacion
            - ids_auditoria_medianas
        ),
        "ids_adicionales": len(
            ids_auditoria_medianas
            - set_ids_catalogo_validacion
        ),
        "pares_observacion_resolucion": (
            pares_medianas_unicos
        ),
        "resoluciones": (
            resoluciones_medianas
        ),
    },
    {
        "auditoria": "interpolacion",
        "filas": len(
            auditoria_interpolacion_validacion
        ),
        "observaciones_unicas": len(
            ids_auditoria_interpolacion
        ),
        "ids_faltantes_catalogo": len(
            set_ids_catalogo_validacion
            - ids_auditoria_interpolacion
        ),
        "ids_adicionales": len(
            ids_auditoria_interpolacion
            - set_ids_catalogo_validacion
        ),
        "pares_observacion_resolucion": np.nan,
        "resoluciones": [],
    },
])


# ------------------------------------------------------------
# Resumen integral
# ------------------------------------------------------------

resumen_validacion = pd.DataFrame([
    {
        "metrica": "Filas del catálogo",
        "valor": len(
            catalogo_validacion
        ),
    },
    {
        "metrica": "Identificadores únicos en el catálogo",
        "valor": (
            catalogo_validacion[
                "observacion_id"
            ].nunique()
        ),
    },
    {
        "metrica": "Estrellas físicas",
        "valor": (
            catalogo_validacion[
                "star_id_base"
            ].nunique()
        ),
    },
    {
        "metrica": "Estrellas con dos observaciones",
        "valor": int(
            catalogo_validacion.loc[
                dos_observaciones_catalogo,
                "star_id_base",
            ].nunique()
        ),
    },
    {
        "metrica": "Pares OGLE conservados",
        "valor": len(
            pares_finales
        ),
    },
    {
        "metrica": "Pares con subtipos diferentes",
        "valor": (
            pares_finales_discordantes
        ),
    },
    {
        "metrica": "Diferencias en dos_observaciones",
        "valor": (
            diferencias_dos_observaciones
        ),
    },
    {
        "metrica": "Diferencias en grupo_analisis",
        "valor": (
            diferencias_grupo_analisis
        ),
    },
    {
        "metrica": "Columnas eliminadas todavía presentes",
        "valor": len(
            columnas_eliminadas_presentes
        ),
    },
    {
        "metrica": "Filas del Parquet de curvas",
        "valor": len(
            ids_curvas_validacion
        ),
    },
    {
        "metrica": "Identificadores únicos en el Parquet",
        "valor": len(
            set_ids_curvas_validacion
        ),
    },
    {
        "metrica": "IDs faltantes en el Parquet",
        "valor": len(
            set_ids_catalogo_validacion
            - set_ids_curvas_validacion
        ),
    },
    {
        "metrica": "IDs adicionales en el Parquet",
        "valor": len(
            set_ids_curvas_validacion
            - set_ids_catalogo_validacion
        ),
    },
    {
        "metrica": "Mismo orden catálogo-Parquet",
        "valor": (
            ids_catalogo_validacion
            == ids_curvas_validacion
        ),
    },
    {
        "metrica": "Mediciones en el Parquet",
        "valor": (
            mediciones_curvas_validacion
        ),
    },
    {
        "metrica": "IDs retirados todavía presentes",
        "valor": len(
            ids_retirar
            & set_ids_curvas_validacion
        ),
    },
])


# ------------------------------------------------------------
# Distribuciones finales
# ------------------------------------------------------------

distribucion_subtipos = (
    catalogo_validacion[
        "subtipo_rrlyrae"
    ]
    .value_counts()
    .rename_axis("subtipo_rrlyrae")
    .reset_index(name="curvas")
)

distribucion_grupos = (
    catalogo_validacion[
        "grupo_analisis"
    ]
    .value_counts()
    .rename_axis("grupo_analisis")
    .reset_index(name="curvas")
)


display(
    resumen_validacion
)

display(
    validacion_datasets
)

display(
    resumen_auditorias_validacion
)

display(
    distribucion_subtipos
)

display(
    distribucion_grupos
)

,metrica,valor
0,Filas del catálogo,102778
1,Identificadores únicos en el catálogo,102778
2,Estrellas físicas,70269
3,Estrellas con dos observaciones,32509
4,Pares OGLE conservados,32509
5,Pares con subtipos diferentes,0
6,Diferencias en dos_observaciones,0
7,Diferencias en grupo_analisis,0
8,Columnas eliminadas todavía presentes,0
9,Filas del Parquet de curvas,102778


,metodo,resolucion,filas,caracteristicas,ids_unicos,ids_duplicados,ids_faltantes_catalogo,ids_adicionales,mismo_orden_catalogo,valores_nan,valores_infinitos,valor_minimo,valor_maximo
0,medianas_fase,50,102778,50,102778,0,0,0,True,0,0,11.874000,21.732500
1,medianas_fase,100,102778,100,102778,0,0,0,True,0,0,11.856501,21.946999
2,interpolacion,50,102778,50,102778,0,0,0,True,0,0,11.863799,22.158365
3,interpolacion,100,102778,100,102778,0,0,0,True,0,0,11.852608,22.411007


,auditoria,filas,observaciones_unicas,ids_faltantes_catalogo,ids_adicionales,pares_observacion_resolucion,resoluciones
0,medianas_fase,205556,102778,0,0,"205,556.000000","[50, 100]"
1,interpolacion,102778,102778,0,0,NaN,[]


,subtipo_rrlyrae,curvas
0,RRab,73600
1,RRc,25626
2,RRd,3353
3,RRe,177
4,aRRd,22


,grupo_analisis,curvas
0,principal,102579
1,minoritaria,199


In [8]:
# ============================================================
# Sustitución de los archivos existentes por los productos
# validados con prioridad de OGLE-IV
# ============================================================

# ------------------------------------------------------------
# Validaciones obligatorias antes de sobrescribir
# ------------------------------------------------------------

errores_promocion = []

if len(catalogo_validacion) != 102_778:
    errores_promocion.append(
        "El catálogo no contiene 102.778 filas."
    )

if (
    catalogo_validacion[
        "observacion_id"
    ].nunique()
    != 102_778
):
    errores_promocion.append(
        "El catálogo no contiene 102.778 "
        "identificadores únicos."
    )

if (
    catalogo_validacion[
        "star_id_base"
    ].nunique()
    != 70_269
):
    errores_promocion.append(
        "El catálogo no contiene 70.269 "
        "estrellas físicas."
    )

if pares_finales_discordantes != 0:
    errores_promocion.append(
        "Todavía existen pares con subtipos diferentes."
    )

if diferencias_dos_observaciones != 0:
    errores_promocion.append(
        "La columna dos_observaciones no es consistente."
    )

if diferencias_grupo_analisis != 0:
    errores_promocion.append(
        "La columna grupo_analisis no es consistente."
    )

if columnas_eliminadas_presentes:
    errores_promocion.append(
        "Todavía existen columnas que debían eliminarse."
    )

if len(ids_curvas_validacion) != 102_778:
    errores_promocion.append(
        "El Parquet no contiene 102.778 curvas."
    )

if mediciones_curvas_validacion != 131_540_677:
    errores_promocion.append(
        "El Parquet no contiene 131.540.677 mediciones."
    )

if (
    ids_catalogo_validacion
    != ids_curvas_validacion
):
    errores_promocion.append(
        "El catálogo y el Parquet no tienen "
        "los mismos identificadores en el mismo orden."
    )

if not (
    validacion_datasets["filas"]
    == 102_778
).all():
    errores_promocion.append(
        "Uno o más datasets no contienen 102.778 filas."
    )

if not (
    validacion_datasets[
        "ids_duplicados"
    ] == 0
).all():
    errores_promocion.append(
        "Uno o más datasets contienen IDs duplicados."
    )

if not (
    validacion_datasets[
        "ids_faltantes_catalogo"
    ] == 0
).all():
    errores_promocion.append(
        "Uno o más datasets tienen IDs faltantes."
    )

if not (
    validacion_datasets[
        "ids_adicionales"
    ] == 0
).all():
    errores_promocion.append(
        "Uno o más datasets tienen IDs adicionales."
    )

if not (
    validacion_datasets[
        "mismo_orden_catalogo"
    ]
).all():
    errores_promocion.append(
        "Uno o más datasets no mantienen "
        "el orden del catálogo."
    )

if not (
    validacion_datasets[
        "valores_nan"
    ] == 0
).all():
    errores_promocion.append(
        "Uno o más datasets contienen valores NaN."
    )

if not (
    validacion_datasets[
        "valores_infinitos"
    ] == 0
).all():
    errores_promocion.append(
        "Uno o más datasets contienen valores infinitos."
    )

if len(
    auditoria_medianas_validacion
) != 205_556:
    errores_promocion.append(
        "La auditoría de medianas no contiene "
        "205.556 filas."
    )

if len(
    auditoria_interpolacion_validacion
) != 102_778:
    errores_promocion.append(
        "La auditoría de interpolación no contiene "
        "102.778 filas."
    )

if errores_promocion:
    detalle = "\n".join(
        f"- {error}"
        for error in errores_promocion
    )

    raise ValueError(
        "No se sobrescribieron archivos porque "
        "la validación encontró problemas:\n"
        f"{detalle}"
    )


# ------------------------------------------------------------
# Verificación de todos los archivos provisionales
# ------------------------------------------------------------

reemplazos = {
    ruta_catalogo_salida: (
        ruta_catalogo_entrada
    ),
    ruta_curvas_salida: (
        ruta_curvas_entrada
    ),
}

for clave in rutas_datasets_salida:
    reemplazos[
        rutas_datasets_salida[clave]
    ] = rutas_datasets_entrada[clave]

for nombre in rutas_auditorias_salida:
    reemplazos[
        rutas_auditorias_salida[nombre]
    ] = rutas_auditorias_entrada[nombre]

archivos_provisionales_faltantes = [
    ruta_provisional
    for ruta_provisional in reemplazos
    if not ruta_provisional.exists()
]

if archivos_provisionales_faltantes:
    detalle = "\n".join(
        str(ruta.resolve())
        for ruta
        in archivos_provisionales_faltantes
    )

    raise FileNotFoundError(
        "Faltan archivos provisionales:\n"
        f"{detalle}"
    )


# ------------------------------------------------------------
# Liberación de referencias a archivos Parquet
#
# Esto evita bloqueos al reemplazar archivos en Windows.
# ------------------------------------------------------------

archivo_curvas = None
archivo_curvas_entrada = None
archivo_curvas_validacion = None

tabla_dataset = None
tabla_dataset_filtrada = None
tabla_auditoria = None
tabla_auditoria_filtrada = None
tabla_ids = None
tabla_grupo = None


# ------------------------------------------------------------
# Sobrescritura de los archivos existentes
#
# Path.replace mueve cada archivo provisional hacia su ruta
# definitiva y reemplaza el archivo que existía anteriormente.
# ------------------------------------------------------------

reemplazos_realizados = []

for ruta_provisional, ruta_definitiva in (
    reemplazos.items()
):
    ruta_provisional.replace(
        ruta_definitiva
    )

    reemplazos_realizados.append({
        "archivo": ruta_definitiva.name,
        "ruta_definitiva": str(
            ruta_definitiva.resolve()
        ),
        "existe": (
            ruta_definitiva.exists()
        ),
    })

reemplazos_realizados = pd.DataFrame(
    reemplazos_realizados
)


# ------------------------------------------------------------
# Eliminación de archivos provisionales adicionales
# ------------------------------------------------------------

if ruta_auditoria_seleccion.exists():
    ruta_auditoria_seleccion.unlink()

# El directorio debería quedar vacío después de mover los
# productos y eliminar la auditoría provisional.
if directorio_salida.exists():
    archivos_restantes = list(
        directorio_salida.iterdir()
    )

    if archivos_restantes:
        detalle = "\n".join(
            str(ruta.resolve())
            for ruta in archivos_restantes
        )

        raise RuntimeError(
            "Los archivos principales fueron reemplazados, "
            "pero el directorio provisional aún contiene:\n"
            f"{detalle}"
        )

    directorio_salida.rmdir()


# ------------------------------------------------------------
# Reapertura mínima de los archivos definitivos
# ------------------------------------------------------------

catalogo_definitivo = pd.read_csv(
    ruta_catalogo_entrada,
    low_memory=False,
)

archivo_curvas_definitivo = pq.ParquetFile(
    ruta_curvas_entrada
)

revision_definitiva = pd.DataFrame([
    {
        "producto": "Catálogo maestro",
        "filas": len(
            catalogo_definitivo
        ),
        "ruta": str(
            ruta_catalogo_entrada.resolve()
        ),
    },
    {
        "producto": "Curvas plegadas",
        "filas": (
            archivo_curvas_definitivo
            .metadata
            .num_rows
        ),
        "ruta": str(
            ruta_curvas_entrada.resolve()
        ),
    },
    {
        "producto": "Medianas por fase, 50",
        "filas": (
            pq.ParquetFile(
                rutas_datasets_entrada[
                    ("medianas_fase", 50)
                ]
            )
            .metadata
            .num_rows
        ),
        "ruta": str(
            rutas_datasets_entrada[
                ("medianas_fase", 50)
            ].resolve()
        ),
    },
    {
        "producto": "Medianas por fase, 100",
        "filas": (
            pq.ParquetFile(
                rutas_datasets_entrada[
                    ("medianas_fase", 100)
                ]
            )
            .metadata
            .num_rows
        ),
        "ruta": str(
            rutas_datasets_entrada[
                ("medianas_fase", 100)
            ].resolve()
        ),
    },
    {
        "producto": "Interpolación, 50",
        "filas": (
            pq.ParquetFile(
                rutas_datasets_entrada[
                    ("interpolacion", 50)
                ]
            )
            .metadata
            .num_rows
        ),
        "ruta": str(
            rutas_datasets_entrada[
                ("interpolacion", 50)
            ].resolve()
        ),
    },
    {
        "producto": "Interpolación, 100",
        "filas": (
            pq.ParquetFile(
                rutas_datasets_entrada[
                    ("interpolacion", 100)
                ]
            )
            .metadata
            .num_rows
        ),
        "ruta": str(
            rutas_datasets_entrada[
                ("interpolacion", 100)
            ].resolve()
        ),
    },
    {
        "producto": "Auditoría de medianas",
        "filas": (
            pq.ParquetFile(
                rutas_auditorias_entrada[
                    "medianas_fase"
                ]
            )
            .metadata
            .num_rows
        ),
        "ruta": str(
            rutas_auditorias_entrada[
                "medianas_fase"
            ].resolve()
        ),
    },
    {
        "producto": "Auditoría de interpolación",
        "filas": (
            pq.ParquetFile(
                rutas_auditorias_entrada[
                    "interpolacion"
                ]
            )
            .metadata
            .num_rows
        ),
        "ruta": str(
            rutas_auditorias_entrada[
                "interpolacion"
            ].resolve()
        ),
    },
])


display(
    reemplazos_realizados
)

display(
    revision_definitiva
)

print(
    "Los archivos anteriores fueron sobrescritos."
)

print(
    "El directorio provisional fue eliminado."
)

,archivo,ruta_definitiva,existe
0,catalogo_maestro_final.csv,C:\Users\adanm\Code\tesis\data\interim\catalog...,True
1,curvas_plegadas_final.parquet,C:\Users\adanm\Code\tesis\data\interim\curvas_...,True
2,medianas_fase_50.parquet,C:\Users\adanm\Code\tesis\data\processed\discr...,True
3,medianas_fase_100.parquet,C:\Users\adanm\Code\tesis\data\processed\discr...,True
4,interpolacion_50.parquet,C:\Users\adanm\Code\tesis\data\processed\discr...,True
5,interpolacion_100.parquet,C:\Users\adanm\Code\tesis\data\processed\discr...,True
6,auditoria_medianas_fase.parquet,C:\Users\adanm\Code\tesis\data\processed\discr...,True
7,auditoria_interpolacion.parquet,C:\Users\adanm\Code\tesis\data\processed\discr...,True


,producto,filas,ruta
0,Catálogo maestro,102778,C:\Users\adanm\Code\tesis\data\interim\catalog...
1,Curvas plegadas,102778,C:\Users\adanm\Code\tesis\data\interim\curvas_...
2,"Medianas por fase, 50",102778,C:\Users\adanm\Code\tesis\data\processed\discr...
3,"Medianas por fase, 100",102778,C:\Users\adanm\Code\tesis\data\processed\discr...
4,"Interpolación, 50",102778,C:\Users\adanm\Code\tesis\data\processed\discr...
5,"Interpolación, 100",102778,C:\Users\adanm\Code\tesis\data\processed\discr...
6,Auditoría de medianas,205556,C:\Users\adanm\Code\tesis\data\processed\discr...
7,Auditoría de interpolación,102778,C:\Users\adanm\Code\tesis\data\processed\discr...


Los archivos anteriores fueron sobrescritos.
El directorio provisional fue eliminado.
